# Poiseuilleov tok: predvidi → izračunaj → provjeri

Za potpuno razvijen laminarni tok newtonskog fluida u kružnoj cijevi profil je paraboličan. Referentni slučaj odgovara sintetičkom analitičkom paketu `data/cfd/poiseuille_laminar`: voda, promjer 20 mm, duljina 1 m, pad tlaka 2,004 Pa. Podatci su navedeni u ćeliji pa pokus radi i u JupyterLiteu bez pristupa datotekama repozitorija.

Razlikujemo pogrešku numeričke integracije, ulaznu mjernu nesigurnost i valjanost fizikalnog modela. Poslije osnovnog pokusa slijede podatci Z3, profili Z4 i izbor mreže Z6. Sintetički skupovi nisu rezultat novoga CFD proračuna ni stvarnog mjerenja.

## Predvidi

1. Hoće li trapezna integracija osnosimetričnog profila precijeniti ili podcijeniti protok? Kako će se pogreška mijenjati udvostručenjem broja intervala?
2. Može li finija mreža opravdati laminarni model pri velikom Reynoldsovu broju?
3. Zašto relativna nesigurnost promjera u Z3 snažno utječe na viskoznost?
4. Hoće li se pri većem nepovoljnom gradijentu u Z4 fluid gibati unatrag na samoj donjoj ploči ili samo uz nju?
5. Mora li mreža s prihvatljivim GCI-jem zadovoljiti i bilancu mase?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 110, "font.size": 10})

radius, delta_p, mu, length, rho = 0.010, 2.004, 1.002e-3, 1.0, 998.2
umax = delta_p*radius**2/(4*mu*length)
q_exact = np.pi*radius**4*delta_p/(8*mu*length)

umean = q_exact/(np.pi*radius**2)
reynolds = rho*umean*(2*radius)/mu
print(f"u_max = {umax:.5f} m/s; Re = {reynolds:.2f}; Q = {q_exact:.9e} m3/s")
assert np.isclose(reynolds, 498.1037924151697, rtol=1e-12)
assert reynolds < 1000  # Ovaj slučaj je duboko u laminarnom rasponu.

def q_trapezoid(n):
    r = np.linspace(0, radius, n+1)
    u = umax*(1-(r/radius)**2)
    return np.trapezoid(2*np.pi*r*u, r)

levels = np.array([8, 16, 32, 64, 128])
q_num = np.array([q_trapezoid(int(n)) for n in levels])
errors = np.abs(q_num-q_exact)
orders = np.log(errors[:-1]/errors[1:])/np.log(2)
print(" n       Q_num (m3/s)       |pogreška|       opaženi red")
for i, n in enumerate(levels):
    order = "-" if i == 0 else f"{orders[i-1]:.3f}"
    print(f"{n:3d}   {q_num[i]:.9f}     {errors[i]:.3e}      {order}")

## Izračunaj: propagacija ulazne nesigurnosti referentnog protoka

Za Poiseuilleov zakon $Q\propto R^4\Delta p/(\mu L)$ linearizirana relativna standardna nesigurnost sadrži član $4u_R/R$. Za ovaj zasebni sintetički pokus pretpostavljamo neovisne normalne ulaze s malim standardnim nesigurnostima; uspoređujemo RSS i Monte Carlo uzorkovanje. To nisu zajamčene granice pogreške.

In [ ]:
sigma_R, sigma_dp, sigma_mu, sigma_L = 0.00004, 0.02004, 1.002e-5, 0.001
relative_linear = np.sqrt(
    (4*sigma_R/radius)**2 + (sigma_dp/delta_p)**2 +
    (sigma_mu/mu)**2 + (sigma_L/length)**2
)
u_q_linear = q_exact*relative_linear

rng = np.random.default_rng(20260803)
n_samples = 40_000
R_mc = rng.normal(radius, sigma_R, n_samples)
dp_mc = rng.normal(delta_p, sigma_dp, n_samples)
mu_mc = rng.normal(mu, sigma_mu, n_samples)
L_mc = rng.normal(length, sigma_L, n_samples)
q_mc = np.pi*R_mc**4*dp_mc/(8*mu_mc*L_mc)
u_q_mc = np.std(q_mc, ddof=1)

contribution_radius = (4*sigma_R/radius)**2/relative_linear**2
print(f"u_Q linearno = {u_q_linear:.3e} m3/s; Monte Carlo = {u_q_mc:.3e} m3/s")
print(f"Polumjer nosi {100*contribution_radius:.1f} % linearizirane varijance.")

## Provjeri

Konvergencija provjerava red numeričke metode. Neovisno provjeravamo analitičku vezu \(Q=A\,u_{max}/2\) i slaganje dvaju postupaka propagacije nesigurnosti.


In [ ]:
assert np.all(q_num < q_exact)
assert np.all(errors[1:] < errors[:-1])
assert np.isclose(umax*(1-(radius/radius)**2), 0.0, atol=1e-15)
assert np.allclose(orders[-3:], 2.0, atol=0.02)
assert np.isclose(q_exact, np.pi*radius**2*umax/2, rtol=1e-14)
assert abs(u_q_mc/u_q_linear-1) < 0.05
assert contribution_radius > 0.5

r = np.linspace(0, radius, 250)
fig, axes = plt.subplots(1, 3, figsize=(12, 3.8))
axes[0].plot(1e3*r, umax*(1-(r/radius)**2), color="#256d85")
axes[0].set(xlabel="r (mm)", ylabel="u (m/s)", title="Analitički profil")
axes[1].loglog(levels, errors, "o-", color="#b43c35")
axes[1].set(xlabel="broj intervala", ylabel="$|Q_h-Q|$", title="Mrežna konvergencija")
axes[2].hist(1e3*q_mc, bins=50, color="#7cb5d6", edgecolor="white")
axes[2].set(xlabel="Q (L/s)", ylabel="broj uzoraka", title="Ulazna nesigurnost")
for ax in axes: ax.grid(True, ls=":", alpha=.45)
plt.tight_layout(); plt.show()

## Z3: obrnuti problem i doprinosi mjernoj nesigurnosti

Koristimo točno podatke zadatka: $Q=0{,}300\pm0{,}003$ mL/min, $\Delta p=652\pm5$ Pa, $L=0{,}200\pm0{,}001$ m, $D=0{,}500\pm0{,}005$ mm i $\rho=998$ kg/m³. Znakovi ± znače neovisne standardne nesigurnosti. Mjerni razmak leži između priključaka na istoj visini u potpuno razvijenom laminarnom toku uz prianjanje na stijenku. Gustoća se smatra točno zadanom.

In [ ]:
Q3, dp3, L3, D3, rho3 = 0.300e-6/60, 652.0, 0.200, 0.500e-3, 998.0
mu3 = np.pi*D3**4*dp3/(128*L3*Q3)
rel_terms = np.array([4*0.005/0.500, 5/652, 0.001/0.200, 0.003/0.300])
u_mu3 = mu3*np.linalg.norm(rel_terms)
Re3 = rho3*(Q3/(np.pi*D3**2/4))*D3/mu3
print(f"mu = {mu3*1e3:.6f} mPa s; u(mu) = {u_mu3*1e3:.6f} mPa s; Re = {Re3:.4f}")
print("Udio D, tlaka, L i Q u varijanci:", rel_terms**2/np.sum(rel_terms**2))
assert abs(mu3*1e3 - 1.000) <= 0.0005
assert abs(u_mu3*1e3 - 0.042) <= 0.0005
assert abs(Re3 - 12.7) <= 0.05
assert np.argmax(rel_terms**2) == 0
# Izračunata viskoznost mora reproducirati zadani izmjereni protok.
assert np.isclose(np.pi*D3**4*dp3/(128*mu3*L3), Q3, rtol=1e-14)

## Z4: prianjanje i lokalni povratni tok

Za $μ=0{,}100$ Pa s, $U=2{,}00$ m/s i $H=1{,}00$ mm prikazujemo profile pri 90 % i 110 % kritičnog pozitivnog gradijenta tlaka. Vodoravna os grafa je **brzina**, a okomita položaj u procjepu: ovo nije crtež strujnica u prostoru. Obje ploče su nepropusne; $τ_0$ je komponenta sile fluida na donju ploču u smjeru gibanja gornje ploče.

In [ ]:
H4, U4, mu4 = 1e-3, 2.0, 0.100
Gcrit = 2*mu4*U4/H4**2
def profile4(y, factor):
    return U4*y/H4 + factor*Gcrit*(y**2-H4*y)/(2*mu4)

y4 = np.linspace(0, H4, 401)
fig, ax = plt.subplots(figsize=(5.8, 4.0))
for factor, color in [(0.9, "#1565c0"), (1.1, "#c0392b")]:
    u4 = profile4(y4, factor)
    tau0 = mu4*U4/H4 - factor*Gcrit*H4/2
    ax.plot(u4, y4*1e3, label=f"G/Gkrit = {factor:.2f}; tau0 = {tau0:+.1f} Pa", color=color)
    assert np.isclose(u4[0], 0.0, atol=1e-14)
    assert np.isclose(u4[-1], U4, atol=1e-14)
assert np.all(profile4(y4[1:], 0.9) > 0)
assert profile4(H4/22, 1.1) < 0
assert np.isclose(profile4(H4/11, 1.1), 0.0, atol=1e-14)
ax.axvline(0, ls="--", color="#3a4a56", lw=1)
ax.set(xlabel="u (m/s)", ylabel="y (mm)", ylim=(0, H4*1e3), title="Z4: dva profila u istom procjepu")
ax.grid(ls=":", alpha=.4); ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

## Z6: GCI i odluka o mreži

Ispisani protoci su zaokruženi sintetički podatci; **računamo s istim brojevima kao student u knjizi**. Koraci su $h/h_f=(4,2,1)$ i $F_s=1{,}25$. GCI procjenjuje diskretizacijsku pogrešku protoka, dok maseni debalans zasebno provjerava očuvanje. Za nastavni izbor vrijede oba uvjeta: GCI ≤ 0,50 % i maseni debalans ≤ 0,0050 %. To nisu univerzalni pragovi ni zamjena za validaciju.

In [ ]:
q6 = np.array([8.16814, 7.93252, 7.87362])*1e-6
mass6 = np.array([0.040, 0.010, 0.0025])  # postotci
h6 = np.array([4.0, 2.0, 1.0])
r6, Fs6 = 2.0, 1.25
p6 = np.log((q6[0]-q6[1])/(q6[1]-q6[2]))/np.log(r6)
qext6 = q6[2] + (q6[2]-q6[1])/(r6**p6-1)
gci6 = 100*Fs6*np.abs(np.diff(q6)/q6[1:])/(r6**p6-1)
acceptable6 = (gci6 <= 0.50) & (mass6[1:] <= 0.0050)
print(f"p = {p6:.9f}; Qext = {qext6:.10e} m3/s")
print("GCI srednje i fine (%):", gci6)
print("Maseni debalans srednje i fine (%):", mass6[1:])
print("Oba uvjeta zadovoljena:", acceptable6)
assert abs(p6 - 2.000) <= 0.0005
assert abs(qext6 - 7.8540e-6) <= 0.00005e-6
assert np.allclose(gci6, [1.237, 0.312], atol=0.0005, rtol=0)
assert np.array_equal(acceptable6, [False, True])
assert mass6[0] > 0.0050  # Gruba je isključena već masenom bilancom.
# Referentni analitički protok provjerava ograničenje sintetičkog skupa.
assert abs(qext6/q_exact - 1) < 2e-6
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(h6**2, q6*1e6, "o-", label="zaokruženi sintetički podatci", color="#1565c0")
ax.plot(0, qext6*1e6, "s", label="ekstrapolacija", color="#c0392b")
ax.axhline(q_exact*1e6, ls="--", color="#1e8449", label="analitička referenca")
ax.set(xlabel="(h/h_f)²", ylabel="Q (mL/s)", title="Z6: približno drugi red konvergencije")
ax.grid(ls=":", alpha=.4); ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

## Protumači

1. Koliko se smanjuje pogreška trapezne integracije kad udvostručiš broj intervala? Zašto njezin predznak nije isti kao predznak odstupanja u zasebnom sintetičkom skupu Z6?
2. Koji podatak u Z3 treba preciznije mjeriti da se najviše smanji nesigurnost viskoznosti? Zašto udio u varijanci nije isto što i relativna standardna nesigurnost?
3. U kojem je dijelu procjepa u Z4 brzina negativna i zašto na donjoj ploči ipak ostaje nula? Dokazuje li to u ovoj potpuno razvijenoj geometriji odvajanje strujnica od ploče?
4. Koja ponuđena mreža zadovoljava oba uvjeta Z6? Daje li ta odluka jamstvo za smično naprezanje na stijenci koje ovdje nije provjeravano?
5. Mogu li preciznija mjerenja ili finija mreža ukloniti pogrešan izbor laminarnog modela? Koji bi podatci bili potrebni za njegovu validaciju?
6. U izvatku `data/cfd/hydrofoil_experiment` nedostaju povijesti reziduala i sila, masena bilanca i potpun popis doprinosa mjernoj nesigurnosti. Koje su usporedbe još moguće, a koji zaključak te praznine onemogućuju?